# Dimensionality Reduction — Exercises

Companion to the note [Dimensionality Reduction](Dimensionality%20Reduction.md).

Practise the methods in the note's table beyond plain PCA: the curse of dimensionality, random projections and Johnson–Lindenstrauss, classical MDS and Isomap, LDA, kernel PCA, and how to judge (and not over-interpret) t-SNE / UMAP embeddings, by hand and from scratch in NumPy, checked against scikit-learn.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_digits, load_iris, make_circles, make_swiss_roll
from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import Isomap, TSNE, trustworthiness
from sklearn.random_projection import johnson_lindenstrauss_min_dim
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Match the method to the goal
*🧠 Concept · ★☆☆*

Using the note's table, pick a method for each task and say what it preserves:

(a) Compress 10 000-dimensional bag-of-words vectors for fast approximate nearest-neighbour search, no training budget.
(b) A 2-D picture of 50 000 single-cell gene profiles for a paper figure.
(c) Reduce features before a classifier when you have labels and 3 classes.
(d) Unroll data lying on a curved 2-D sheet in 3-D (Swiss roll).
(e) Denoise images by keeping the main linear structure.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: linear or non-linear? supervised or not? for visualization only, or reusable as features for new data?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Random projection**: data-independent, fast, approximately preserves pairwise distances (Johnson–Lindenstrauss).
(b) **t-SNE / UMAP**: preserve local neighbourhoods, made for visualization (UMAP scales better and keeps a bit more global structure).
(c) **LDA**: supervised, at most $C-1=2$ components that maximize class separation.
(d) **Isomap** (geodesic distances) or another manifold method, maybe kernel PCA.
(e) **PCA** ([[PCA]]): keep the top components and reconstruct; noise lives mostly in the low-variance directions.

</details>

### Exercise 2 · Reading a t-SNE plot
*🧠 Concept · ★☆☆*

A colleague shows a t-SNE plot and concludes: (1) "cluster A is three times bigger than B, so A is more diverse",
(2) "A and C are far apart, so they are very different", (3) "there are clearly 7 groups". Which conclusions are justified?
What would you do to check them?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Re-read the warning box in the note. t-SNE optimizes the match of *neighbour probabilities*, with an adaptive bandwidth per point.

</details>

<details>
<summary><b>✅ Solution</b></summary>

None are reliable. t-SNE adapts a bandwidth per point (perplexity), which equalizes densities, so **cluster sizes** mean little.
Its heavy-tailed low-dimensional kernel pushes clusters apart by arbitrary amounts, so **between-cluster distances** are not meaningful.
The **number of clumps** depends on perplexity, the random seed and iterations; t-SNE can even show clusters in pure noise.
Checks: vary perplexity and seed, compare with PCA, compute cluster statistics in the original space, use a quantitative measure such as trustworthiness (Exercise 16) or clustering metrics on the original data ([[Clustering]]).

</details>

### Exercise 3 · The curse of dimensionality
*🧠 Concept · ★★☆*

List three distinct effects behind the "curse of dimensionality" mentioned in the note, and explain for each how it hurts a
specific algorithm (e.g. k-NN, density estimation, grid search over input space).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think volume (how much data you need), distances (contrast between near and far), and where the volume of a cube/ball sits.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Data sparsity / sample complexity**: covering $[0,1]^d$ with cells of width $0.1$ needs $10^d$ cells, so histograms and kernel density estimates need exponentially many samples.
2. **Distance concentration**: for random points, $(\max d-\min d)/\min d\to0$ as $d$ grows (Exercise 10), so "nearest" neighbours are hardly nearer than the rest and k-NN ([[k-Nearest Neighbors]]) or RBF kernels lose meaning.
3. **Volume near the boundary**: almost all of a high-dimensional cube's volume lies near its surface (Exercise 5) and a Gaussian's mass sits in a thin shell at radius $\approx\sqrt d$, so intuitions about "typical" points fail and extrapolation is the norm.
Remedies: dimensionality reduction, feature selection, strong priors/regularization, exploiting that real data lies near low-dimensional manifolds.

</details>

### Exercise 4 · PCA vs LDA
*🧠 Concept · ★★☆*

Both PCA and LDA are linear projections. State the objective each maximizes, how many components each can return for
$d$ features and $C$ classes, and sketch a 2-D data set where the first PCA direction is useless for classification
but the LDA direction is perfect.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

LDA maximizes $\frac{w^\top S_B w}{w^\top S_W w}$; $S_B$ has rank at most $C-1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

PCA maximizes the total variance $w^\top S w$ (unsupervised), up to $\min(N,d)$ components. LDA maximizes the Fisher ratio of between-class to within-class scatter, $\frac{w^\top S_Bw}{w^\top S_Ww}$ (supervised), at most $C-1$ components because $\operatorname{rank}S_B\le C-1$.
Example: two long, thin, parallel cigar-shaped classes along the $x$-axis, separated by a small gap in $y$. PCA picks $x$ (huge variance), where the classes overlap completely; LDA picks $y$, where they separate perfectly. See [[Linear Models for Classification]].

</details>

## Part B · By hand

### Exercise 5 · Where is the volume of a hypercube?
*✍️ By hand · ★☆☆*

What fraction of the unit hypercube $[0,1]^d$ lies within distance $0.05$ of its boundary (i.e. **outside** the inner cube $[0.05,0.95]^d$)?
Compute it for $d=10$ and $d=100$.

In [ ]:
frac_d10 = None
frac_d100 = None

_u = rng.random((200000, 10)); _mc = np.mean(((_u < .05) | (_u > .95)).any(1))
check('d=10 (vs Monte Carlo)', frac_d10, _mc, tol=5e-3)
check('d=100', frac_d100, 1 - 0.9 ** 100)

<details>
<summary><b>💡 Hint</b></summary>

The inner cube has side $0.9$, so volume $0.9^d$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$1-0.9^{10}\approx0.651$ and $1-0.9^{100}\approx0.99997$. In 100 dimensions essentially every uniformly drawn point is near the boundary.

```python
frac_d10 = 1 - 0.9 ** 10
frac_d100 = 1 - 0.9 ** 100
```

</details>

### Exercise 6 · Johnson–Lindenstrauss target dimension
*✍️ By hand · ★☆☆*

The JL lemma guarantees that a random projection to $k\ge \dfrac{4\ln n}{\varepsilon^2/2-\varepsilon^3/3}$ dimensions preserves all
pairwise squared distances among $n$ points within a factor $1\pm\varepsilon$ (with high probability).
Compute the bound for $n=1000$ points and $\varepsilon=0.1$ (round down to an integer like sklearn). What is remarkable about it?

In [ ]:
k_jl = None

check('JL bound', k_jl, johnson_lindenstrauss_min_dim(1000, eps=0.1))

<details>
<summary><b>💡 Hint</b></summary>

$\ln 1000\approx6.908$; $\varepsilon^2/2-\varepsilon^3/3=0.005-0.000333$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$k\ge 4\cdot6.908/0.004667\approx5920.9$, so $5920$ (sklearn rounds down).
Remarkable: the bound does **not depend on the original dimension $d$** at all, only (logarithmically) on $n$. It is useful when $d$ is huge (e.g. $10^6$); for $\varepsilon=0.1$ it is quite conservative, and in practice far fewer dimensions often suffice.

```python
k_jl = 5920
```

</details>

### Exercise 7 · Classical MDS by hand
*✍️ By hand · ★★☆*

Three points on a line have pairwise distances $d_{12}=1,\ d_{13}=3,\ d_{23}=2$. Classical MDS forms
$B=-\tfrac12 JD^{(2)}J$ with $D^{(2)}_{ij}=d_{ij}^2$ and centring matrix $J=I-\tfrac1n\mathbf 1\mathbf 1^\top$, then uses the top eigenvectors scaled by $\sqrt{\lambda}$.

Show that $B$ is the Gram matrix $X_cX_c^\top$ of the centred coordinates, and compute its largest eigenvalue. How many non-zero eigenvalues does it have?

In [ ]:
lam_max_B = None
n_nonzero = None

_D2 = np.array([[0, 1, 9], [1, 0, 4], [9, 4, 0.]]); _J = np.eye(3) - 1/3
_l = np.linalg.eigvalsh(-0.5 * _J @ _D2 @ _J)
check('largest eigenvalue of B', lam_max_B, _l.max())
check('number of non-zero eigenvalues', n_nonzero, int(np.sum(np.abs(_l) > 1e-9)))

<details>
<summary><b>💡 Hint 1</b></summary>

Put the points at $0,1,3$ (consistent with the distances), then centre them: mean is $4/3$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For 1-D centred coordinates $x$, $B=xx^\top$ has a single non-zero eigenvalue $\lVert x\rVert^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$d_{ij}^2=\lVert x_i\rVert^2+\lVert x_j\rVert^2-2x_i^\top x_j$; double centring kills the first two terms (they are constant along rows/columns) and leaves $B=X_cX_c^\top$.
Centred coordinates: $(-\tfrac43,-\tfrac13,\tfrac53)$, so $B=xx^\top$ with eigenvalue $\lVert x\rVert^2=\tfrac{16+1+25}{9}=\tfrac{42}9\approx4.667$, and only **one** non-zero eigenvalue: the configuration is 1-dimensional. The embedding $\sqrt{\lambda}\,u$ recovers $x$ up to sign.
With Euclidean distances classical MDS is identical to PCA on the coordinates; Isomap uses geodesic graph distances instead.

```python
lam_max_B = 42/9
n_nonzero = 1
```

</details>

### Exercise 8 · Fisher's LDA direction
*✍️ By hand · ★★☆*

For two classes, the Fisher direction is $w\propto S_W^{-1}(m_1-m_2)$. With within-class scatter
$S_W=\begin{pmatrix}2&0\\0&0.5\end{pmatrix}$ and $m_1-m_2=(1,1)^\top$, compute the unit-length $w$. Compare it with the direction $m_1-m_2$ and explain.

In [ ]:
w_lda = None  # unit vector

_w = np.linalg.solve(np.diag([2, .5]), [1., 1]); _w /= np.linalg.norm(_w)
check('LDA direction (up to sign)', None if w_lda is None else np.abs(w_lda), np.abs(_w))

<details>
<summary><b>💡 Hint</b></summary>

$S_W^{-1}=\mathrm{diag}(0.5,2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$S_W^{-1}(m_1-m_2)=(0.5,2)$; normalized $w=(0.5,2)/\sqrt{4.25}\approx(0.243,0.970)$.
The mean difference alone would be $(1,1)/\sqrt2$, but feature 1 is noisy within classes (variance 2) while feature 2 is precise (0.5). LDA tilts the direction towards the low-noise feature, maximizing separation relative to spread.

```python
w_lda = np.array([0.5, 2]) / np.linalg.norm([0.5, 2])
```

</details>

### Exercise 9 · Centring in feature space (kernel PCA)
*✍️ By hand · ★★★*

Kernel PCA ([[Kernel Methods]]) runs PCA on $\phi(x_i)$ without computing $\phi$. Show that the Gram matrix of the **centred**
features $\tilde\phi_i=\phi(x_i)-\frac1N\sum_j\phi(x_j)$ is
$$\tilde K = K-\mathbf 1_NK-K\mathbf 1_N+\mathbf 1_NK\mathbf 1_N,\qquad (\mathbf 1_N)_{ij}=\tfrac1N.$$
Then explain why the projection of a training point on component $j$ is $\sqrt{\lambda_j}\,a_{ij}$ where $\tilde K a_j=\lambda_ja_j$, $\lVert a_j\rVert=1$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Expand $\tilde\phi_i^\top\tilde\phi_l$ into four inner products.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The feature-space eigenvector is $v_j=\sum_i \alpha_{ij}\tilde\phi_i$; unit length requires $\alpha_j=a_j/\sqrt{\lambda_j}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\tilde\phi_i^\top\tilde\phi_l=K_{il}-\frac1N\sum_jK_{jl}-\frac1N\sum_jK_{ij}+\frac1{N^2}\sum_{j,m}K_{jm}$, which is exactly the matrix formula.
Write $\tilde\Phi$ (rows $\tilde\phi_i$), so $\tilde K=\tilde\Phi\tilde\Phi^\top$. If $\tilde Ka=\lambda a$, then $v=\tilde\Phi^\top a/\sqrt\lambda$ satisfies $\lVert v\rVert^2=a^\top\tilde Ka/\lambda=1$ and is an eigenvector of the feature-space covariance.
The projection of training point $i$ is $\tilde\phi_i^\top v=(\tilde K a)_i/\sqrt\lambda=\sqrt\lambda\,a_i$. This is the same "$U\Sigma$" relation as PCA via SVD.

</details>

## Part C · Code

### Exercise 10 · Distance concentration
*💻 Code · ★☆☆*

For $d\in\{2,10,100,1000\}$ draw 1000 uniform points in $[0,1]^d$ and one query point, compute the Euclidean distances and the
relative contrast $(d_{\max}-d_{\min})/d_{\min}$. Store the four values in `contrast`. What happens?

In [ ]:
dims = [2, 10, 100, 1000]
contrast = None

check('contrast shrinks with dimension', None if contrast is None else bool(np.all(np.diff(contrast) < 0) and contrast[-1] < 0.3), True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.norm(P - q, axis=1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The contrast goes from about 50 (2-D) to about 0.1 (1000-D): all points are roughly equally far from the query. Distances concentrate because the squared distance is a sum of $d$ independent terms: its mean grows like $d$ but its standard deviation only like $\sqrt d$.

```python
dims = [2, 10, 100, 1000]
contrast = []
for d in dims:
    P = rng.random((1000, d)); q = rng.random(d)
    dist = np.linalg.norm(P - q, axis=1)
    contrast.append((dist.max() - dist.min()) / dist.min())
print(np.round(contrast, 3))
```

</details>

### Exercise 11 · Gaussian random projection
*💻 Code · ★★☆*

Implement `random_project(X, k, rng)`: multiply by a $d\times k$ matrix $R$ with i.i.d. $\mathcal N(0,1/k)$ entries.
Project 200 points from $d=5000$ to $k=1000$ and compute the ratio of projected to original **squared** pairwise distances.
Store the fraction of pairs whose ratio is within $[0.9, 1.1]$ in `frac_within`.

In [ ]:
def random_project(X, k, rng):
    # TODO
    return None

X_hd = rng.normal(size=(200, 5000))
frac_within = None

check('most distances preserved within 10%', None if frac_within is None else frac_within > 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

With entries of variance $1/k$, $\mathbb E\lVert R^\top x\rVert^2=\lVert x\rVert^2$. Use `pdist` to get all pairwise distances.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each squared projected length is a scaled $\chi^2_k$ variable with relative std $\sqrt{2/k}\approx0.045$, so almost all ratios fall in $[0.9,1.1]$ — about 98% here.
No training was needed and the cost is one matrix multiply; sparse random matrices (Achlioptas) make it even cheaper.

```python
def random_project(X, k, rng):
    R = rng.normal(scale=1 / np.sqrt(k), size=(X.shape[1], k))
    return X @ R

X_hd = rng.normal(size=(200, 5000))
ratio = pdist(random_project(X_hd, 1000, rng), 'sqeuclidean') / pdist(X_hd, 'sqeuclidean')
frac_within = np.mean((ratio > 0.9) & (ratio < 1.1))
print(frac_within, ratio.min(), ratio.max())
```

</details>

### Exercise 12 · Classical MDS from scratch
*💻 Code · ★★☆*

Implement `classical_mds(D, k)` from a distance matrix $D$ (Exercise 7): double centring, `eigh`, top-$k$ eigenvectors times $\sqrt\lambda$.
Recover a 2-D configuration of 30 random points from their distance matrix only, and verify that the distances are reproduced exactly.

In [ ]:
def classical_mds(D, k=2):
    # TODO
    return None

P2 = rng.normal(size=(30, 2))
Y_mds = classical_mds(squareform(pdist(P2)), 2)

check('distances reproduced', None if Y_mds is None else pdist(Y_mds), pdist(P2))

<details>
<summary><b>💡 Hint</b></summary>

`J = np.eye(n) - 1/n`; `B = -0.5 * J @ (D**2) @ J`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The recovered configuration equals the original up to a rotation/reflection and translation, which is all a distance matrix can determine.

```python
def classical_mds(D, k=2):
    n = len(D)
    J = np.eye(n) - 1 / n
    B = -0.5 * J @ (D ** 2) @ J
    lam, V = np.linalg.eigh(B)
    lam, V = lam[::-1][:k], V[:, ::-1][:, :k]
    return V * np.sqrt(np.maximum(lam, 0))

P2 = rng.normal(size=(30, 2))
Y_mds = classical_mds(squareform(pdist(P2)), 2)
```

</details>

### Exercise 13 · Kernel PCA from scratch
*💻 Code · ★★★*

Implement RBF kernel PCA with $K_{ij}=\exp(-\gamma\lVert x_i-x_j\rVert^2)$, $\gamma=2$, using the centring formula and projection
$\sqrt{\lambda_j}a_{ij}$ from Exercise 9. Apply it to two concentric circles and compare with `KernelPCA`. Then fit a logistic regression on
the **first** kernel component and on the raw 2-D inputs, and store both training accuracies (`acc_kpca`, `acc_raw`).

In [ ]:
X_circ, y_circ = make_circles(200, factor=0.3, noise=0.05, random_state=0)
def kernel_pca(X, k=2, gamma=2.0):
    # TODO
    return None

Z_kpca = kernel_pca(X_circ)
acc_kpca = None
acc_raw = None

check('matches sklearn KernelPCA (up to sign)', None if Z_kpca is None else np.abs(Z_kpca), np.abs(KernelPCA(2, kernel='rbf', gamma=2.0).fit_transform(X_circ)), tol=1e-6)
check('one kernel component separates the rings', None if acc_kpca is None else (acc_kpca > 0.98 and acc_raw < 0.7), True)

<details>
<summary><b>💡 Hint</b></summary>

`one = np.ones((N, N)) / N`; `Kc = K - one @ K - K @ one + one @ K @ one`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A linear classifier on the raw circles is at chance (about 50%), while the first RBF kernel component essentially encodes the radius and separates the rings perfectly.
The bandwidth matters: with $\gamma=0.5$ or $1$ the first components do not separate the rings, with $\gamma=10$ one component is no longer enough. Kernel PCA needs the $N\times N$ kernel matrix, so it scales as $O(N^2)$ memory and $O(N^3)$ time.

```python
X_circ, y_circ = make_circles(200, factor=0.3, noise=0.05, random_state=0)
def kernel_pca(X, k=2, gamma=2.0):
    N = len(X)
    K = np.exp(-gamma * ((X[:, None] - X[None]) ** 2).sum(-1))
    one = np.ones((N, N)) / N
    Kc = K - one @ K - K @ one + one @ K @ one
    lam, A = np.linalg.eigh(Kc)
    lam, A = lam[::-1][:k], A[:, ::-1][:, :k]
    return A * np.sqrt(lam)

Z_kpca = kernel_pca(X_circ)
acc_kpca = LogisticRegression().fit(Z_kpca[:, :1], y_circ).score(Z_kpca[:, :1], y_circ)
acc_raw = LogisticRegression().fit(X_circ, y_circ).score(X_circ, y_circ)
print(acc_kpca, acc_raw)
```

</details>

### Exercise 14 · LDA from scratch on iris
*💻 Code · ★★☆*

Implement multi-class LDA: $S_W=\sum_c\sum_{i\in c}(x_i-m_c)(x_i-m_c)^\top$, $S_B=\sum_c N_c(m_c-m)(m_c-m)^\top$, and take the top
$C-1=2$ eigenvectors of $S_W^{-1}S_B$. Project iris to 2-D and compare the first component with sklearn's `LinearDiscriminantAnalysis`
(store the absolute correlation in `corr_lda1`). Also plot PCA vs LDA projections side by side.

In [ ]:
iris = load_iris(); Xi, yi = iris.data, iris.target
def lda_fit(X, y, k=2):
    # TODO: return W (d, k)
    return None

W_lda = lda_fit(Xi, yi)
corr_lda1 = None

check('first LDA component matches sklearn', corr_lda1, 1.0, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.eig(np.linalg.solve(S_W, S_B))` (not symmetric, so `eig`; take the real parts and sort by eigenvalue).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The components agree up to scale and sign (correlation 1). On iris, LDA's first component alone almost perfectly separates the three species, while PCA's PC1 mixes versicolor and virginica more, because PCA ignores the labels.
Only 2 non-zero eigenvalues exist since $\operatorname{rank}S_B\le C-1$.

```python
import matplotlib.pyplot as plt
iris = load_iris(); Xi, yi = iris.data, iris.target
def lda_fit(X, y, k=2):
    m = X.mean(0)
    d = X.shape[1]
    S_W, S_B = np.zeros((d, d)), np.zeros((d, d))
    for c in np.unique(y):
        Xc = X[y == c]; mc = Xc.mean(0)
        S_W += (Xc - mc).T @ (Xc - mc)
        S_B += len(Xc) * np.outer(mc - m, mc - m)
    lam, V = np.linalg.eig(np.linalg.solve(S_W, S_B))
    order = np.argsort(lam.real)[::-1]
    return V[:, order[:k]].real

W_lda = lda_fit(Xi, yi)
Z_l = Xi @ W_lda
Z_sk = LinearDiscriminantAnalysis(n_components=2).fit_transform(Xi, yi)
corr_lda1 = abs(np.corrcoef(Z_l[:, 0], Z_sk[:, 0])[0, 1])
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].scatter(*PCA(2).fit_transform(Xi).T, c=yi, s=10); ax[0].set_title('PCA')
ax[1].scatter(*Z_l.T, c=yi, s=10); ax[1].set_title('LDA (scratch)')
plt.show()
```

</details>

### Exercise 15 · Unrolling the Swiss roll: PCA vs Isomap
*💻 Code · ★★☆*

`make_swiss_roll(1000, random_state=0)` returns points $X\in\mathbb R^{3}$ and the position $t$ along the roll. Embed in 2-D with PCA and
with `Isomap(n_neighbors=10)`, and store the absolute Spearman correlation between the first embedding coordinate and $t$
in `rho_pca` and `rho_isomap`. What does Isomap do differently, and what happens with too many neighbours?

In [ ]:
X_roll, t_roll = make_swiss_roll(1000, random_state=0)
rho_pca = None
rho_isomap = None

check('Isomap recovers the roll parameter', None if rho_isomap is None else (rho_isomap > 0.95 and rho_isomap > rho_pca + 0.3), True)

<details>
<summary><b>💡 Hint</b></summary>

`abs(spearmanr(a, b)[0])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Isomap builds a k-nearest-neighbour graph, replaces Euclidean distances by shortest-path (geodesic) distances along the graph, and then runs classical MDS (Exercise 12). Geodesics follow the sheet, so its first coordinate is essentially $t$ ($\rho\approx1$); PCA only sees the 3-D shape and projects the roll flat ($\rho\approx0.2$).
With too many neighbours the graph "short-circuits" between layers of the roll and the unrolling breaks.

```python
X_roll, t_roll = make_swiss_roll(1000, random_state=0)
rho_pca = abs(spearmanr(PCA(2).fit_transform(X_roll)[:, 0], t_roll)[0])
rho_isomap = abs(spearmanr(Isomap(n_neighbors=10, n_components=2).fit_transform(X_roll)[:, 0], t_roll)[0])
print(rho_pca, rho_isomap)
```

</details>

### Exercise 16 · Quantifying an embedding: trustworthiness
*💻 Code · ★★☆*

Trustworthiness $\in[0,1]$ measures how many of the $k$ nearest neighbours **in the embedding** were also near in the original space.
On the first 500 digits, compute it (`n_neighbors=10`) for a 2-D PCA embedding and a 2-D t-SNE embedding (`TSNE(2, random_state=0, init='pca')`),
store `tw_pca` and `tw_tsne`, and plot both embeddings coloured by digit.

In [ ]:
dig = load_digits(); Xd, yd = dig.data[:500], dig.target[:500]
tw_pca = None
tw_tsne = None

check('t-SNE preserves local neighbourhoods better', None if tw_tsne is None else (tw_tsne > tw_pca and tw_tsne > 0.95), True)

<details>
<summary><b>💡 Hint</b></summary>

`trustworthiness(X, Z, n_neighbors=10)` from `sklearn.manifold`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

t-SNE scores about 0.99 vs about 0.86 for PCA: local neighbourhoods are kept far better, which is exactly what it optimizes.
But trustworthiness says nothing about global geometry (distances between clusters), which t-SNE does not preserve. Also, t-SNE has no `transform` for new points, so it is for visualization, not a feature extractor.

```python
import matplotlib.pyplot as plt
dig = load_digits(); Xd, yd = dig.data[:500], dig.target[:500]
Zp = PCA(2).fit_transform(Xd)
Zt = TSNE(2, random_state=0, init='pca').fit_transform(Xd)
tw_pca = trustworthiness(Xd, Zp, n_neighbors=10)
tw_tsne = trustworthiness(Xd, Zt, n_neighbors=10)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(*Zp.T, c=yd, cmap='tab10', s=8); ax[0].set_title(f'PCA  T={tw_pca:.2f}')
ax[1].scatter(*Zt.T, c=yd, cmap='tab10', s=8); ax[1].set_title(f't-SNE  T={tw_tsne:.2f}')
plt.show()
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Dimensionality Reduction](Dimensionality%20Reduction.md).*